# Effective barrier height $\Phi$ and exchange splitting $\Delta_{\mathrm{ex}}$ at 20 K (v2)

This is a clean rebuild of the original `Barrier_vs_canting_angle_20K` notebook with two methodological changes:

1. **Peak fitter.** The conduction-band-edge peak in the Feenstra-normalised differential conductance $(dI/dV)/(I/V)$ is fit with a Gaussian on a linear background, restricted to a window centred on a rough peak estimate. This replaces the local parabolic fit. The asymmetric peak shape biases a parabola, so its vertex sits systematically ~10-30 meV above the true peak; the Gaussian + linear is the more honest estimator.
2. **$\Delta_{\mathrm{ex}}$ extraction.** Both endpoints, $\Phi_{\mathrm{FM}}$ and $\Phi_{\mathrm{AFM}}$, are taken from inverse-variance-weighted constant fits in clean field windows ($|H_z| > 2$ T saturated, $|H_z| < 0.1$ T pure AFM for the $c$-axis; $|H_y| > 0.5$ T and $|H_y| < 0.2$ T for the $b$-axis). $\Delta_{\mathrm{ex}} = \Phi_{\mathrm{AFM}} - \Phi_{\mathrm{FM}}$ with errors in quadrature.

The original $\sin^2(\theta/2)$ linear fit is retained for the $c$-axis as a diagnostic — it tests how well the alternating-barrier ansatz $\Phi(\theta) = \Phi_{\mathrm{FM}} + \Delta_{\mathrm{ex}}\sin^2(\theta/2)$ describes the canting data — but the headline $\Delta_{\mathrm{ex}}$ reported per axis is from the constant fits.


## 1. Setup and data loading

In [ ]:
from scripts.utils import setup_notebook, OKABE_ITO_CYCLE
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

from scripts.IV_Hscan_gaussian import load_dataframe

TEMPERATURE = 20
df_c_path = PROJECT_ROOT / "output" / "IV_H_scans" / "dataframes" / "c_scans" / f"IV_gaussian_{TEMPERATURE}K.pkl"
df_b_path = PROJECT_ROOT / "output" / "IV_H_scans" / "dataframes" / "b_scans" / f"IV_gaussian_{TEMPERATURE}K.pkl"
df_c = load_dataframe(df_c_path)
df_b = load_dataframe(df_b_path)

print(f"c-axis: {len(df_c)} rows, H range {df_c['H'].min():+.3f} to {df_c['H'].max():+.3f} T")
print(f"b-axis: {len(df_b)} rows, H range {df_b['H'].min():+.3f} to {df_b['H'].max():+.3f} T")


## 2. Canting model ($c$-axis)

For A-type \ce{CrSBr} with easy axis along $b$ and field applied along the hard $c$-axis, both sublattice magnetisations cant symmetrically toward $c$. Minimising the interlayer exchange against the Zeeman term gives $\sin\theta_c = H_z / H_{\mathrm{sat}}$ for the canting angle of each sublattice, so the interlayer angle between the two sublattices is

$$\theta(H_z) = 2\,\arccos\!\left(|H_z| / H_{\mathrm{sat}}\right),$$

with $H_{\mathrm{sat}} \approx 2$ T from SQUID magnetometry. The alternating-barrier weight is $\sin^2(\theta/2) = 1 - (H_z/H_{\mathrm{sat}})^2$.


In [ ]:
H_SAT_C = 2.0  # T; c-axis saturation field from SQUID

def canting_angle(H_z, H_sat=H_SAT_C):
    h = np.clip(np.abs(H_z) / H_sat, 0.0, 1.0)
    return 2.0 * np.arccos(h)

def sin2_half(H_z, H_sat=H_SAT_C):
    h = np.clip(np.abs(H_z) / H_sat, 0.0, 1.0)
    return 1.0 - h**2


## 3. Peak fitter: Gaussian + linear background

For each I(V) curve we compute $(dI/dV)/(I/V)$ on the smoothed current, locate a rough peak in a coarse window via Savitzky-Golay smoothing + argmax, then fit

$$f(V) = A\,\exp\!\left[-\tfrac{(V-V_0)^2}{2\sigma^2}\right] + m V + c$$

over $V_{\mathrm{rough}} \pm \texttt{half\_width}$. The peak position $V_{\mathrm{peak}}$ is the numerical maximum of $f$ on a dense grid (essentially $V_0$ when the linear term is small), and its 1-$\sigma$ uncertainty is taken from the covariance-derived error on $V_0$. The fit is rejected if any constrained parameter sits at its bound.

Parameters were chosen by a small grid search (`tmp/playground_gauss_fit/.../v3_grid.py`): `half_width = 0.25 V`, `sigma_bounds = (0.03, 0.30) V`. With these values, both $c$-axis and $b$-axis 20 K datasets give median fit uncertainty $\sim 3$-$5$ meV with $\leq 1$ rejection per axis.


In [ ]:
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

HALF_WIDTH    = 0.25
SIGMA_BOUNDS  = (0.03, 0.30)
COARSE_WINDOW = (0.40, 0.98)
SMOOTH_WIN, SMOOTH_POLY = 21, 3
TRIM, V_MIN = 2, 0.05


def normalised_didv(V, I, trim=TRIM, v_min=V_MIN):
    order = np.argsort(V)
    V, I = np.asarray(V)[order], np.asarray(I)[order]
    dIdV = np.gradient(I, V)
    V, I, dIdV = V[trim:-trim], I[trim:-trim], dIdV[trim:-trim]
    mask = np.abs(V) > v_min
    return V[mask], dIdV[mask] / (I[mask] / V[mask])


def gauss_lin(V, A, V0, sigma, m, c):
    return A * np.exp(-0.5 * ((V - V0) / sigma) ** 2) + m * V + c


def fit_band_edge_gauss(V, norm,
                        coarse=COARSE_WINDOW,
                        half_width=HALF_WIDTH,
                        sigma_bounds=SIGMA_BOUNDS):
    '''Two-step Gaussian + linear-background peak fit.

    Step 1: savgol-smooth (dI/dV)/(I/V) in `coarse` and take argmax -> V_rough.
    Step 2: fit gauss + linear in V_rough +/- half_width, with sigma bounded
            to keep the fit away from runaway-width solutions.

    Returns dict with V_peak (numerical maximum of the model), V_peak_err
    (covariance error on V0), and `ok` set to False if any constrained
    parameter ends up at its bound.
    '''
    m = (V >= coarse[0]) & (V <= coarse[1])
    if m.sum() < SMOOTH_WIN + 2:
        return dict(ok=False, V_peak=np.nan, V_peak_err=np.nan,
                    sigma=np.nan, popt=None, window=None)
    Vf, Nf = V[m], norm[m]
    wl = SMOOTH_WIN if SMOOTH_WIN <= len(Nf) and SMOOTH_WIN % 2 == 1 else max(5, (len(Nf) // 2) * 2 - 1)
    Ns = savgol_filter(Nf, window_length=wl, polyorder=SMOOTH_POLY)
    V_rough = float(Vf[int(np.argmax(Ns))])

    m2 = (Vf >= V_rough - half_width) & (Vf <= V_rough + half_width)
    Vp, Np_ = Vf[m2], Nf[m2]
    if Vp.size < 8:
        return dict(ok=False, V_peak=np.nan, V_peak_err=np.nan,
                    sigma=np.nan, popt=None, window=None, V_rough=V_rough)

    n_edge = max(3, len(Vp) // 6)
    m_init = (Np_[-n_edge:].mean() - Np_[:n_edge].mean()) / (Vp[-n_edge:].mean() - Vp[:n_edge].mean())
    c_init = Np_[:n_edge].mean() - m_init * Vp[:n_edge].mean()
    A_init = max(float(Np_.max() - (m_init * V_rough + c_init)), 1e-3)
    p0 = [A_init, V_rough, 0.06, m_init, c_init]
    lower = [0,                     V_rough - half_width, sigma_bounds[0], -np.inf, -np.inf]
    upper = [10 * abs(A_init) + 10, V_rough + half_width, sigma_bounds[1],  np.inf,  np.inf]

    try:
        popt, pcov = curve_fit(gauss_lin, Vp, Np_, p0=p0, bounds=(lower, upper),
                               maxfev=30000)
    except Exception:
        return dict(ok=False, V_peak=np.nan, V_peak_err=np.nan,
                    sigma=np.nan, popt=None, window=None, V_rough=V_rough)

    perr = np.sqrt(np.diag(pcov))
    A_fit, V0_fit, sigma_fit, _, _ = popt
    eps = 1e-3
    at_bound = (sigma_fit < sigma_bounds[0] + eps or sigma_fit > sigma_bounds[1] - eps
                or V0_fit < V_rough - half_width + eps
                or V0_fit > V_rough + half_width - eps
                or A_fit < eps)

    Vgrid = np.linspace(Vp.min(), Vp.max(), 4001)
    V_peak = float(Vgrid[int(np.argmax(gauss_lin(Vgrid, *popt)))])
    return dict(ok=not at_bound, V_peak=V_peak, V_peak_err=float(perr[1]),
                sigma=float(sigma_fit), popt=popt,
                window=(float(Vp.min()), float(Vp.max())), V_rough=V_rough)


## 4. Representative dI/dV traces with peak fits

Spot-check the fitter at a handful of fields to confirm the peak position and width evolve sensibly with $H$. $c$-axis: continuous canting from AFM ($H_z = 0$) to saturated FM ($|H_z| \geq H_{\mathrm{sat}}$). $b$-axis: sharp spin-flip near $H_y \sim 0.3$ T separating the two endpoint states.


In [ ]:
def plot_spot_check(df, target_fields, axis_label):
    fig, ax = plt.subplots(figsize=(6, 5), dpi=600)
    peak_lines = []
    for i, H_target in enumerate(target_fields):
        idx = (df['H'] - H_target).abs().idxmin()
        row = df.loc[idx]
        V, norm = normalised_didv(row['voltage_smooth'], row['current_smooth'])
        g = fit_band_edge_gauss(V, norm)
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        ax.plot(V, norm, 'o', color=color, markersize=2.5, alpha=0.45,
                label=fr'${axis_label}={row["H"]:+.2f}$ T')
        if g['popt'] is not None:
            Vmin, Vmax = g['window']
            Vfit = np.linspace(Vmin, Vmax, 400)
            ax.plot(Vfit, gauss_lin(Vfit, *g['popt']), '-', color=color, lw=1.8)
        if g['ok'] and np.isfinite(g['V_peak']):
            ax.axvline(g['V_peak'], color=color, ls='--', lw=1.0, alpha=0.85)
            peak_lines.append(
                fr'${axis_label}={row["H"]:+.2f}$ T: '
                fr'$\Phi=({1000 * g["V_peak"]:.0f}\pm{1000 * g["V_peak_err"]:.0f})$ meV'
            )
    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel(r'$(dI/dV)/(I/V)$')
    ax.set_xlim(-0.05, 1.0)
    ax.legend(loc='upper left', fontsize=15, frameon=True)
    fig.tight_layout()
    plt.show()


plot_spot_check(df_c, [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0],
                axis_label='H_Z')


In [ ]:
plot_spot_check(df_b, [0.0, 0.10, 0.15, 0.50, 0.70, 1.00],
                axis_label='H_y')


## 5. Extract $V_{\mathrm{peak}}$ for every $I(V)$ curve

Apply the Gaussian + linear fitter to every curve in both dataframes. Curves where any constrained fit parameter ends up at a bound are dropped — those points are typically dominated by the transition region where no clean local peak exists.


In [ ]:
def extract_peaks(df):
    rows = []
    for _, r in df.iterrows():
        V, norm = normalised_didv(r['voltage_smooth'], r['current_smooth'])
        g = fit_band_edge_gauss(V, norm)
        rows.append({
            'H':          r['H'],
            'Phi_eV':     g['V_peak'],
            'Phi_err_eV': g['V_peak_err'],
            'sigma':      g['sigma'],
            'ok':         g['ok'],
        })
    out = pd.DataFrame(rows)
    return out[out['ok']].reset_index(drop=True)


res_c = extract_peaks(df_c)
res_b = extract_peaks(df_b)
res_c['abs_H'] = res_c['H'].abs()
res_b['abs_H'] = res_b['H'].abs()
res_c['theta_deg'] = np.degrees(canting_angle(res_c['H'].values))
res_c['sin2_half'] = sin2_half(res_c['H'].values)

print(f"c-axis: {len(res_c)} / {len(df_c)} curves accepted (median uncertainty {1000 * res_c['Phi_err_eV'].median():.2f} meV)")
print(f"b-axis: {len(res_b)} / {len(df_b)} curves accepted (median uncertainty {1000 * res_b['Phi_err_eV'].median():.2f} meV)")


## 6. Helpers: inverse-variance weighted mean and binning

In [ ]:
def weighted_mean(values, errors):
    '''Inverse-variance weighted mean with Birge-rescaled SEM.

    Bins with zero/non-finite errors are assigned the median positive error
    so a single noisy point cannot dominate. If no point has positive error,
    falls back to the sample standard deviation.

    The SEM 1/sqrt(sum w_i) assumes the per-point errors fully account for
    the scatter. If chi^2_red = (1/(n-1)) * sum w_i (v_i - mu)^2 exceeds 1,
    the per-point errors are under-reported (or the data are over-dispersed
    relative to a true constant), and the SEM is inflated by sqrt(chi^2_red)
    (Birge ratio / PDG convention). chi^2_red < 1 is not used to shrink the
    SEM.
    '''
    v = np.asarray(values, float)
    e = np.asarray(errors, float)
    good = np.isfinite(v) & np.isfinite(e)
    v, e = v[good], e[good]
    if v.size == 0:
        return np.nan, np.nan, 0
    pos = e > 0
    if pos.any():
        e = np.where(pos, e, np.median(e[pos]))
    else:
        e = np.full_like(v, np.std(v, ddof=1) if v.size > 1 else 1.0)
    w = 1.0 / e**2
    mu       = float(np.sum(w * v) / np.sum(w))
    sigma_mu = float(1.0 / np.sqrt(np.sum(w)))
    if v.size > 1:
        chi2_red = float(np.sum(w * (v - mu)**2) / (v.size - 1))
        if chi2_red > 1.0:
            sigma_mu *= np.sqrt(chi2_red)
    return mu, sigma_mu, int(v.size)


def bin_weighted(df, key, round_decimals=2):
    '''Group `df` by `df[key].round(round_decimals)`, then weighted-average
    Phi_eV inside each bin. Returns one row per bin.
    '''
    out = []
    for _, sub in df.groupby(df[key].round(round_decimals)):
        mu, sig, n = weighted_mean(sub['Phi_eV'], sub['Phi_err_eV'])
        out.append({key: float(sub[key].mean()),
                    'Phi_eV': mu, 'Phi_err_eV': sig, 'n': n})
    return pd.DataFrame(out).sort_values(key).reset_index(drop=True)

## 7. $c$-axis $\Phi(H_z)$ with endpoint constant fits

$\Phi(H_z)$ is symmetric about $H_z = 0$, maximal at the AFM state, and saturates to $\Phi_{\mathrm{FM}}$ for $|H_z| \geq H_{\mathrm{sat}}$. The two horizontal bands show the inverse-variance-weighted constant fits over $|H_z| < 0.1$ T (AFM endpoint) and $|H_z| > 2$ T (saturated endpoint).


In [ ]:
H_AFM_C_MAX = 0.10   # T -- pure AFM (no observable canting in binned data)
H_FM_C_MIN  = 2.10   # T -- saturated FM-aligned

afm_c_mask = res_c['abs_H'] < H_AFM_C_MAX
fm_c_mask  = res_c['abs_H'] > H_FM_C_MIN

Phi_AFM_c, Phi_AFM_c_err, n_AFM_c = weighted_mean(res_c.loc[afm_c_mask, 'Phi_eV'],
                                                  res_c.loc[afm_c_mask, 'Phi_err_eV'])
Phi_FM_c,  Phi_FM_c_err,  n_FM_c  = weighted_mean(res_c.loc[fm_c_mask,  'Phi_eV'],
                                                  res_c.loc[fm_c_mask,  'Phi_err_eV'])
Delta_c     = Phi_AFM_c - Phi_FM_c
Delta_c_err = float(np.sqrt(Phi_AFM_c_err**2 + Phi_FM_c_err**2))

# Bin the whole sweep for plotting (10 mT resolution).
agg_c_H = bin_weighted(res_c, 'H', round_decimals=2)

fig, ax = plt.subplots(figsize=(6, 5), dpi=600)
excl_c = ~(afm_c_mask | fm_c_mask)
ax.errorbar(res_c.loc[afm_c_mask, 'H'], 1000 * res_c.loc[afm_c_mask, 'Phi_eV'],
            yerr=1000 * res_c.loc[afm_c_mask, 'Phi_err_eV'],
            fmt='o', color=OKABE_ITO_CYCLE[1], markersize=4, alpha=0.85,
            ecolor=OKABE_ITO_CYCLE[1], elinewidth=0.7, capsize=2,
            label=fr'AFM ($|H_z|<{H_AFM_C_MAX}$ T), $n={n_AFM_c}$')
ax.errorbar(res_c.loc[fm_c_mask, 'H'], 1000 * res_c.loc[fm_c_mask, 'Phi_eV'],
            yerr=1000 * res_c.loc[fm_c_mask, 'Phi_err_eV'],
            fmt='s', color=OKABE_ITO_CYCLE[5], markersize=4, alpha=0.85,
            ecolor=OKABE_ITO_CYCLE[5], elinewidth=0.7, capsize=2,
            label=fr'FM ($|H_z|>{H_FM_C_MIN}$ T), $n={n_FM_c}$')
ax.errorbar(res_c.loc[excl_c, 'H'], 1000 * res_c.loc[excl_c, 'Phi_eV'],
            yerr=1000 * res_c.loc[excl_c, 'Phi_err_eV'],
            fmt='x', color='0.55', markersize=4, alpha=0.55,
            ecolor='0.55', elinewidth=0.5, capsize=1, label='canting (excluded)')

# Constant-fit horizontal bands on their data ranges.
for sign in (-1, +1):
    Hs_fm = res_c.loc[fm_c_mask & (np.sign(res_c['H']) == sign), 'H']
    if not Hs_fm.empty:
        ax.hlines(1000 * Phi_FM_c, Hs_fm.min(), Hs_fm.max(),
                  color=OKABE_ITO_CYCLE[5], lw=1.8)
        ax.fill_between([Hs_fm.min(), Hs_fm.max()],
                        1000 * (Phi_FM_c - Phi_FM_c_err),
                        1000 * (Phi_FM_c + Phi_FM_c_err),
                        color=OKABE_ITO_CYCLE[5], alpha=0.25, lw=0)
Hs_afm = res_c.loc[afm_c_mask, 'H']
if not Hs_afm.empty:
    ax.hlines(1000 * Phi_AFM_c, Hs_afm.min(), Hs_afm.max(),
              color=OKABE_ITO_CYCLE[1], lw=1.8)
    ax.fill_between([Hs_afm.min(), Hs_afm.max()],
                    1000 * (Phi_AFM_c - Phi_AFM_c_err),
                    1000 * (Phi_AFM_c + Phi_AFM_c_err),
                    color=OKABE_ITO_CYCLE[1], alpha=0.25, lw=0)

ax.set_xlabel(r'$H_Z$ (T)')
ax.set_ylabel(r'$\Phi$ (meV)')
ax.legend(loc='lower right', fontsize=8)
ax.text(0.03, 0.97,
        fr'$\Phi_{{\mathrm{{AFM}}}}=({1000 * Phi_AFM_c:.1f}\pm{1000 * Phi_AFM_c_err:.1f})$ meV''\n'
        fr'$\Phi_{{\mathrm{{FM}}}}\,=\,({1000 * Phi_FM_c:.1f}\pm{1000 * Phi_FM_c_err:.1f})$ meV''\n'
        fr'$\Delta_{{\mathrm{{ex}}}}\,=\,({1000 * Delta_c:.1f}\pm{1000 * Delta_c_err:.1f})$ meV',
        transform=ax.transAxes, va='top', ha='left', fontsize=10,
        bbox=dict(facecolor='white', edgecolor='0.7', alpha=0.9))
fig.tight_layout()
plt.show()

print(f"c-axis Phi_AFM  = ({1000 * Phi_AFM_c:.2f} +/- {1000 * Phi_AFM_c_err:.2f}) meV  (n={n_AFM_c})")
print(f"c-axis Phi_FM   = ({1000 * Phi_FM_c:.2f}  +/- {1000 * Phi_FM_c_err:.2f}) meV  (n={n_FM_c})")
print(f"c-axis Delta_ex = ({1000 * Delta_c:.2f}  +/- {1000 * Delta_c_err:.2f}) meV")


In [ ]:
# Publication-ready version of the panel above: same physics, but each
# field bin from `agg_c_H` is shown as a single inverse-variance-weighted
# point so the figure is not dominated by sweep redundancy.

abs_H_bin   = agg_c_H['H'].abs()
afm_bin     = abs_H_bin < H_AFM_C_MAX
fm_bin      = abs_H_bin > H_FM_C_MIN
excl_bin    = ~(afm_bin | fm_bin)

fig, ax = plt.subplots(figsize=(6, 5), dpi=600)

ax.errorbar(agg_c_H.loc[afm_bin, 'H'], 1000 * agg_c_H.loc[afm_bin, 'Phi_eV'],
            yerr=1000 * agg_c_H.loc[afm_bin, 'Phi_err_eV'],
            fmt='o', color=OKABE_ITO_CYCLE[1], markersize=5.5,
            ecolor=OKABE_ITO_CYCLE[1], elinewidth=0.9, capsize=2.5,
            label=fr'AFM ($|H_Z|<{H_AFM_C_MAX}$ T)')
ax.errorbar(agg_c_H.loc[fm_bin, 'H'], 1000 * agg_c_H.loc[fm_bin, 'Phi_eV'],
            yerr=1000 * agg_c_H.loc[fm_bin, 'Phi_err_eV'],
            fmt='s', color=OKABE_ITO_CYCLE[5], markersize=5.5,
            ecolor=OKABE_ITO_CYCLE[5], elinewidth=0.9, capsize=2.5,
            label=fr'FM ($|H_Z|>{H_FM_C_MIN}$ T)')
ax.errorbar(agg_c_H.loc[excl_bin, 'H'], 1000 * agg_c_H.loc[excl_bin, 'Phi_eV'],
            yerr=1000 * agg_c_H.loc[excl_bin, 'Phi_err_eV'],
            fmt='x', color='0.45', markersize=5.5,
            ecolor='0.45', elinewidth=0.7, capsize=2,
            label='Canted state')

# Constant-fit horizontal bands over their data ranges.
for sign in (-1, +1):
    Hs_fm = agg_c_H.loc[fm_bin & (np.sign(agg_c_H['H']) == sign), 'H']
    if not Hs_fm.empty:
        ax.hlines(1000 * Phi_FM_c, Hs_fm.min(), Hs_fm.max(),
                  color=OKABE_ITO_CYCLE[5], lw=2.0)
        ax.fill_between([Hs_fm.min(), Hs_fm.max()],
                        1000 * (Phi_FM_c - Phi_FM_c_err),
                        1000 * (Phi_FM_c + Phi_FM_c_err),
                        color=OKABE_ITO_CYCLE[5], alpha=0.25, lw=0)
Hs_afm = agg_c_H.loc[afm_bin, 'H']
if not Hs_afm.empty:
    ax.hlines(1000 * Phi_AFM_c, Hs_afm.min(), Hs_afm.max(),
              color=OKABE_ITO_CYCLE[1], lw=2.0)
    ax.fill_between([Hs_afm.min(), Hs_afm.max()],
                    1000 * (Phi_AFM_c - Phi_AFM_c_err),
                    1000 * (Phi_AFM_c + Phi_AFM_c_err),
                    color=OKABE_ITO_CYCLE[1], alpha=0.25, lw=0)

ax.set_xlabel(r'$H_Z$ (T)')
ax.set_ylabel(r'$\Phi$ (meV)')
ax.legend(loc='upper right', fontsize=13)
fig.tight_layout()
plt.show()

print(f"Plotted {len(agg_c_H)} H-bins (10 mT resolution), "
      f"collapsed from {len(res_c)} accepted curves.")


## 8. $\Phi$ versus interlayer canting angle $\theta$

Symmetrising in $|H_z|$ collapses the two field polarities onto a single $\Phi(\theta)$ curve.


In [ ]:
# Clip |H_z| to H_sat so the saturated points all collapse into the theta = 0 bin.
res_c_pos = res_c.copy()
res_c_pos['abs_H_eff'] = np.minimum(res_c_pos['abs_H'].values, H_SAT_C)
agg_c_theta = bin_weighted(res_c_pos.assign(abs_H=res_c_pos['abs_H_eff']),
                           'abs_H', round_decimals=2)
agg_c_theta['theta_deg'] = np.degrees(canting_angle(agg_c_theta['abs_H'].values))
agg_c_theta['sin2_half'] = sin2_half(agg_c_theta['abs_H'].values)

fig, ax = plt.subplots(figsize=(6, 5), dpi=600)
ax.errorbar(agg_c_theta['theta_deg'], 1000 * agg_c_theta['Phi_eV'],
            yerr=1000 * agg_c_theta['Phi_err_eV'],
            fmt='o', color=OKABE_ITO_CYCLE[2], markersize=4,
            ecolor=OKABE_ITO_CYCLE[2], elinewidth=0.8, capsize=2, alpha=0.9)
ax.set_xlabel(r'Interlayer angle $\theta$ (deg)')
ax.set_ylabel(r'$\Phi(\theta)$ (meV)')
ax.set_xticks([0, 45, 90, 135, 180])
fig.tight_layout()
plt.show()

print(f"{len(agg_c_theta)} theta bins")


## 9. Alternating-barrier test: $\Phi$ versus $\sin^2(\theta/2)$

The simplest alternating-barrier construction predicts $\Phi(\theta) = \Phi_{\mathrm{FM}} + \Delta_{\mathrm{ex}}\,\sin^2(\theta/2)$, a straight line. Plotting against $\sin^2(\theta/2)$ together with the constant-fit endpoints reveals whether the ansatz actually holds.

If the line passes through both endpoint bands, the model is consistent. If not, the curvature you see is real, and the constant-fit $\Delta_{\mathrm{ex}}$ should be quoted in preference to the slope of the line.


In [ ]:
# Inverse-variance weighted linear regression in sin^2(theta/2).
x    = agg_c_theta['sin2_half'].values
y    = agg_c_theta['Phi_eV'].values
yerr = agg_c_theta['Phi_err_eV'].values
pos  = yerr > 0
safe = np.where(pos, yerr, np.median(yerr[pos]) if pos.any() else 1.0)
w    = 1.0 / safe**2

S, Sx  = w.sum(), (w * x).sum()
Sxx    = (w * x * x).sum()
Sy, Sxy = (w * y).sum(), (w * x * y).sum()
det = S * Sxx - Sx * Sx
a_lin = (Sxx * Sy - Sx * Sxy) / det          # intercept = Phi_FM
b_lin = (S * Sxy - Sx * Sy) / det            # slope     = Delta_ex
var_a, var_b, cov_ab = Sxx / det, S / det, -Sx / det

# Birge rescale: if chi^2_red > 1, inflate the covariance by chi^2_red so the
# parameter errors reflect the actual scatter of the binned points about the
# line. chi^2_red < 1 is not used to shrink.
resid = y - (a_lin + b_lin * x)
dof = max(x.size - 2, 1)
chi2_red_lin = float(np.sum(w * resid**2) / dof)
scale = max(1.0, chi2_red_lin)
var_a  *= scale
var_b  *= scale
cov_ab *= scale

Phi_FM_c_lin       = a_lin
Phi_FM_c_lin_err   = float(np.sqrt(var_a))
Delta_c_lin        = b_lin
Delta_c_lin_err    = float(np.sqrt(var_b))
Phi_AFM_c_lin      = a_lin + b_lin
Phi_AFM_c_lin_err  = float(np.sqrt(var_a + var_b + 2 * cov_ab))

xx = np.linspace(0.0, 1.0, 200)
yy = a_lin + b_lin * xx
band = np.sqrt(var_a + 2 * xx * cov_ab + xx**2 * var_b)

fig, ax = plt.subplots(figsize=(6, 5), dpi=600)
ax.errorbar(x, 1000 * y, yerr=1000 * yerr,
            fmt='o', color=OKABE_ITO_CYCLE[2], markersize=4, alpha=0.9,
            ecolor=OKABE_ITO_CYCLE[2], elinewidth=0.8, capsize=2, label='c-axis data')
ax.plot(xx, 1000 * yy, '-', color='black', lw=1.3,
        label=fr'Linear fit')
ax.fill_between(xx, 1000 * (yy - band), 1000 * (yy + band),
                color='black', alpha=0.12, lw=0)


ax.set_xlabel(r'$\sin^2(\theta/2) = 1 - (H_Z/H_{\mathrm{sat}})^2$')
ax.set_ylabel(r'$\Phi(\theta)$ (meV)')
ax.set_xlim(-0.05, 1.05)
ax.legend(loc='lower right', fontsize=14)
fig.tight_layout()
plt.show()

print('--- c-axis: linear fit in sin^2(theta/2) (diagnostic) ---')
print(f'  chi^2_red   = {chi2_red_lin:.2f}  (Birge scale applied: {scale:.2f})')
print(f'  Phi_FM_lin  = ({1000 * Phi_FM_c_lin:.2f} +/- {1000 * Phi_FM_c_lin_err:.2f}) meV')
print(f'  Phi_AFM_lin = ({1000 * Phi_AFM_c_lin:.2f} +/- {1000 * Phi_AFM_c_lin_err:.2f}) meV')
print(f'  Delta_lin   = ({1000 * Delta_c_lin:.2f} +/- {1000 * Delta_c_lin_err:.2f}) meV')

## 10. $b$-axis two-state analysis

Along the easy $b$-axis the field-driven transition is a sharp spin-flip near $H_y^{\mathrm{sf}} \sim 0.3$ T, so only two magnetic configurations are sampled. We exclude the transition window $0.2\,\mathrm{T} \le |H_y| \le 0.5\,\mathrm{T}$ and fit constants in each clean window.


In [ ]:
H_AFM_B_MAX = 0.20
H_FM_B_MIN  = 0.50

afm_b_mask = res_b['abs_H'] < H_AFM_B_MAX
fm_b_mask  = res_b['abs_H'] > H_FM_B_MIN

Phi_AFM_b, Phi_AFM_b_err, n_AFM_b = weighted_mean(res_b.loc[afm_b_mask, 'Phi_eV'],
                                                  res_b.loc[afm_b_mask, 'Phi_err_eV'])
Phi_FM_b,  Phi_FM_b_err,  n_FM_b  = weighted_mean(res_b.loc[fm_b_mask,  'Phi_eV'],
                                                  res_b.loc[fm_b_mask,  'Phi_err_eV'])
Delta_b     = Phi_AFM_b - Phi_FM_b
Delta_b_err = float(np.sqrt(Phi_AFM_b_err**2 + Phi_FM_b_err**2))

fig, ax = plt.subplots(figsize=(6, 5), dpi=600)
excl_b = ~(afm_b_mask | fm_b_mask)
ax.errorbar(res_b.loc[afm_b_mask, 'H'], 1000 * res_b.loc[afm_b_mask, 'Phi_eV'],
            yerr=1000 * res_b.loc[afm_b_mask, 'Phi_err_eV'],
            fmt='o', color=OKABE_ITO_CYCLE[1], markersize=4, alpha=0.85,
            ecolor=OKABE_ITO_CYCLE[1], elinewidth=0.7, capsize=2,
            label=fr'AFM ($|H_y|<{H_AFM_B_MAX}$ T), $n={n_AFM_b}$')
ax.errorbar(res_b.loc[fm_b_mask, 'H'], 1000 * res_b.loc[fm_b_mask, 'Phi_eV'],
            yerr=1000 * res_b.loc[fm_b_mask, 'Phi_err_eV'],
            fmt='s', color=OKABE_ITO_CYCLE[5], markersize=4, alpha=0.85,
            ecolor=OKABE_ITO_CYCLE[5], elinewidth=0.7, capsize=2,
            label=fr'FM ($|H_y|>{H_FM_B_MIN}$ T), $n={n_FM_b}$')
ax.errorbar(res_b.loc[excl_b, 'H'], 1000 * res_b.loc[excl_b, 'Phi_eV'],
            yerr=1000 * res_b.loc[excl_b, 'Phi_err_eV'],
            fmt='x', color='0.55', markersize=4, alpha=0.55,
            ecolor='0.55', elinewidth=0.5, capsize=1, label='spin-flip (excluded)')

for sign in (-1, +1):
    ax.axvspan(sign * H_AFM_B_MAX, sign * H_FM_B_MIN, color='0.85', alpha=0.45, lw=0)

for mask, mu, sig, color in [(afm_b_mask, Phi_AFM_b, Phi_AFM_b_err, OKABE_ITO_CYCLE[1]),
                              (fm_b_mask, Phi_FM_b, Phi_FM_b_err, OKABE_ITO_CYCLE[5])]:
    Hs = res_b.loc[mask, 'H']
    for branch in [Hs[Hs < 0], Hs[Hs > 0]]:
        if branch.empty:
            continue
        x0, x1 = branch.min(), branch.max()
        ax.hlines(1000 * mu, x0, x1, color=color, lw=1.8)
        ax.fill_between([x0, x1], 1000 * (mu - sig), 1000 * (mu + sig),
                        color=color, alpha=0.25, lw=0)

ax.set_xlabel(r'$H_y$ (T)')
ax.set_ylabel(r'$\Phi$ (meV)')
ax.legend(loc='lower right', fontsize=8)
ax.text(0.03, 0.97,
        fr'$\Phi_{{\mathrm{{AFM}}}}=({1000 * Phi_AFM_b:.1f}\pm{1000 * Phi_AFM_b_err:.1f})$ meV''\n'
        fr'$\Phi_{{\mathrm{{FM}}}}\,=\,({1000 * Phi_FM_b:.1f}\pm{1000 * Phi_FM_b_err:.1f})$ meV''\n'
        fr'$\Delta_{{\mathrm{{ex}}}}\,=\,({1000 * Delta_b:.1f}\pm{1000 * Delta_b_err:.1f})$ meV',
        transform=ax.transAxes, va='top', ha='left', fontsize=10,
        bbox=dict(facecolor='white', edgecolor='0.7', alpha=0.9))
fig.tight_layout()
plt.show()

print(f"b-axis Phi_AFM  = ({1000 * Phi_AFM_b:.2f} +/- {1000 * Phi_AFM_b_err:.2f}) meV  (n={n_AFM_b})")
print(f"b-axis Phi_FM   = ({1000 * Phi_FM_b:.2f} +/- {1000 * Phi_FM_b_err:.2f}) meV  (n={n_FM_b})")
print(f"b-axis Delta_ex = ({1000 * Delta_b:.2f} +/- {1000 * Delta_b_err:.2f}) meV")


## 11. Axis comparison

In [ ]:
def _q(mu, sig):
    return f'({1000 * mu:7.2f} +/- {1000 * sig:5.2f})'

print('                          Phi_AFM (meV)        Phi_FM (meV)         Delta_ex (meV)')
print(f'  c-axis (const fit)      {_q(Phi_AFM_c, Phi_AFM_c_err)}    {_q(Phi_FM_c, Phi_FM_c_err)}    {_q(Delta_c, Delta_c_err)}')
print(f'  c-axis (sin^2 linear)   {_q(Phi_AFM_c_lin, Phi_AFM_c_lin_err)}    {_q(Phi_FM_c_lin, Phi_FM_c_lin_err)}    {_q(Delta_c_lin, Delta_c_lin_err)}')
print(f'  b-axis (const fit)      {_q(Phi_AFM_b, Phi_AFM_b_err)}    {_q(Phi_FM_b, Phi_FM_b_err)}    {_q(Delta_b, Delta_b_err)}')
print()
diff_meV    = 1000 * abs(Delta_c - Delta_b)
combined_meV = 1000 * float(np.sqrt(Delta_c_err**2 + Delta_b_err**2))
print(f'  |Delta_c - Delta_b| (const fits) = {diff_meV:.2f} meV  '
      f'(combined 1-sigma = {combined_meV:.2f} meV  -> {diff_meV / combined_meV:.1f}-sigma)')


## 12. Save deliverables

Mirrors the v1 output structure under a `barrier_vs_canting_v2` subtree so v1 results are preserved untouched.


In [ ]:
out_c = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'barrier_vs_canting_v2' / 'c_scans'
out_b = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'barrier_vs_canting_v2' / 'b_scans'
out_c.mkdir(parents=True, exist_ok=True)
out_b.mkdir(parents=True, exist_ok=True)

# c-axis: per-bin Phi(theta) table
agg_c_out = agg_c_theta.copy()
agg_c_out['Phi_meV']     = 1000 * agg_c_out['Phi_eV']
agg_c_out['Phi_meV_err'] = 1000 * agg_c_out['Phi_err_eV']
agg_c_out.to_csv(out_c / f'Phi_vs_theta_{TEMPERATURE}K.csv', index=False)

# c-axis: fit parameters
pd.DataFrame([{
    'temperature_K':         TEMPERATURE,
    'H_sat_T':               H_SAT_C,
    'H_AFM_max_T':           H_AFM_C_MAX,
    'H_FM_min_T':            H_FM_C_MIN,
    'n_AFM':                 n_AFM_c,
    'n_FM':                  n_FM_c,
    'Phi_AFM_meV':           1000 * Phi_AFM_c,
    'Phi_AFM_err_meV':       1000 * Phi_AFM_c_err,
    'Phi_FM_meV':            1000 * Phi_FM_c,
    'Phi_FM_err_meV':        1000 * Phi_FM_c_err,
    'Delta_ex_meV':          1000 * Delta_c,
    'Delta_ex_err_meV':      1000 * Delta_c_err,
    'Phi_AFM_lin_meV':       1000 * Phi_AFM_c_lin,
    'Phi_AFM_lin_err_meV':   1000 * Phi_AFM_c_lin_err,
    'Phi_FM_lin_meV':        1000 * Phi_FM_c_lin,
    'Phi_FM_lin_err_meV':    1000 * Phi_FM_c_lin_err,
    'Delta_ex_lin_meV':      1000 * Delta_c_lin,
    'Delta_ex_lin_err_meV':  1000 * Delta_c_lin_err,
}]).to_csv(out_c / f'Phi_vs_theta_fit_{TEMPERATURE}K.csv', index=False)

# b-axis: per-curve table + fit parameters
res_b_out = res_b.copy()
res_b_out['Phi_meV']     = 1000 * res_b_out['Phi_eV']
res_b_out['Phi_meV_err'] = 1000 * res_b_out['Phi_err_eV']
res_b_out['window'] = np.where(afm_b_mask, 'AFM',
                       np.where(fm_b_mask, 'FM', 'excluded'))
res_b_out.to_csv(out_b / f'Phi_vs_Hy_{TEMPERATURE}K.csv', index=False)

pd.DataFrame([{
    'temperature_K':    TEMPERATURE,
    'H_AFM_max_T':      H_AFM_B_MAX,
    'H_FM_min_T':       H_FM_B_MIN,
    'n_AFM':            n_AFM_b,
    'n_FM':             n_FM_b,
    'Phi_AFM_meV':      1000 * Phi_AFM_b,
    'Phi_AFM_err_meV':  1000 * Phi_AFM_b_err,
    'Phi_FM_meV':       1000 * Phi_FM_b,
    'Phi_FM_err_meV':   1000 * Phi_FM_b_err,
    'Delta_ex_meV':     1000 * Delta_b,
    'Delta_ex_err_meV': 1000 * Delta_b_err,
}]).to_csv(out_b / f'Phi_two_state_fit_{TEMPERATURE}K.csv', index=False)

print('Saved:')
for p in [out_c / f'Phi_vs_theta_{TEMPERATURE}K.csv',
          out_c / f'Phi_vs_theta_fit_{TEMPERATURE}K.csv',
          out_b / f'Phi_vs_Hy_{TEMPERATURE}K.csv',
          out_b / f'Phi_two_state_fit_{TEMPERATURE}K.csv']:
    print(f'  {p}')
